# 05 — Rolling Forecast

Expanding-window rolling forecast (Jan 2014 – Dec 2024) for models M0, M2, M3, M4 using LightGBM with balanced class weights.

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
from panelsplit.cross_validation import PanelSplit

warnings.filterwarnings("ignore", category=UserWarning)

P   = Path("../data/processed")
OUT = Path("../data/processed")

In [2]:
panel = pd.read_parquet(P / "panel_with_features.parquet")
panel = panel.sort_values(["date", "state_code"]).reset_index(drop=True)

TARGET = "shock_sahm_05"

# One-hot encode state fixed effects and month-of-year dummies on the full panel
# (same 51 states + 12 months always present → no column mismatch between train/test)
state_dummies = pd.get_dummies(panel["state_code"], prefix="state", dtype=int)
month_dummies = pd.get_dummies(panel["month_of_year"], prefix="moy", dtype=int)

panel_enc = pd.concat([panel, state_dummies, month_dummies], axis=1)

STATE_COLS = state_dummies.columns.tolist()
MOY_COLS   = month_dummies.columns.tolist()

print(f"Panel shape:  {panel_enc.shape}")
print(f"State dummies: {len(STATE_COLS)}   Month dummies: {len(MOY_COLS)}")
print(f"Date range:   {panel_enc.date.min().strftime('%Y-%m')} → {panel_enc.date.max().strftime('%Y-%m')}")

Panel shape:  (8415, 113)
State dummies: 51   Month dummies: 12
Date range:   2011-04 → 2024-12


## Feature set definitions

Following Ben's structure: define each feature list explicitly as variables, not magic strings.

In [3]:
# ── M0: persistence + fixed effects ──────────────────────────────────────────
FEATURES_M0 = (
    ["ur_lag1"]
    + STATE_COLS    # 51 state dummies (one-hot)
    + MOY_COLS      # 12 month-of-year dummies
)

# ── M2: M0 + full AR lags + FRED macro controls ───────────────────────────────
AR_LAGS  = ["ur_lag2", "ur_lag3", "ur_lag6", "ur_lag12", "ur_chg1", "ur_chg3", "ur_std3"]
FRED_LAGS = ["unrate_nat_lag1", "icsa_mean_lag1", "vix_mean_lag1",
             "nasdaq_logret_lag1", "t10y2y_mean_lag1"]

FEATURES_M2 = FEATURES_M0 + AR_LAGS + FRED_LAGS

# ── M3: M2 + state-level leading indicators ───────────────────────────────────
STATE_LEADING = ["log_claims_lag1", "payrolls_yoy_lag1"]

FEATURES_M3 = FEATURES_M2 + STATE_LEADING

# ── M4: M3 + Google Trends (text) + PCA component (fit inside loop) ───────────
GT_TERMS = [
    "file_for_unemployment", "jobs_hiring", "layoffs",
    "resume", "unemployment", "unemployment_benefits"
]
GT_FEATURES = []
for t in GT_TERMS:
    GT_FEATURES += [f"gt_{t}_z", f"gt_{t}_d1", f"gt_{t}_std3", f"gt_{t}_yoy"]

# z-scored levels used as PCA input (PCA is fit on training data only inside the loop)
PCA_INPUT_COLS = [f"gt_{t}_z" for t in GT_TERMS]

# M4 features: M3 + all text features + placeholder for PCA_1 (added dynamically inside loop)
FEATURES_M4_BASE = FEATURES_M3 + GT_FEATURES

SINCE_VARS = ["months_since_last_sahm_shock", "cumulative_sahm_shocks_24m", "sahm_gap_lag1"]
# since-variables are already in M3 via... actually the spec places them in Group C
# Let's confirm they are included in M2+ via the "AR lags" group expansion
# Re-reading spec: since-vars (C) are NOT listed in M0/M2/M3 feature specs explicitly.
# M2 = M0 + AR lags 2-12 + change + rolling std + FRED macro lags
# M3 = M2 + state claims lag + state payrolls lag
# M4 = M3 + Google Trends + PCA
# Since-variables (C) are listed as features in section 4 but not in the model specs M0-M4.
# Decision: add them to M2 as additional AR-type regressors (they describe AR state of the cycle).
FEATURES_M2 = FEATURES_M0 + AR_LAGS + SINCE_VARS + FRED_LAGS
FEATURES_M3 = FEATURES_M2 + STATE_LEADING
FEATURES_M4_BASE = FEATURES_M3 + GT_FEATURES

for name, feats in [("M0", FEATURES_M0), ("M2", FEATURES_M2),
                    ("M3", FEATURES_M3), ("M4_base", FEATURES_M4_BASE)]:
    print(f"{name:8s}: {len(feats):3d} features")

M0      :  64 features
M2      :  79 features
M3      :  81 features
M4_base : 105 features


## Rolling forecast loop

Adapted from Ben's `03_panel_forecasting.ipynb`. Key differences:
- LightGBM (`class_weight='balanced'`) instead of logistic regression
- Four model specifications run per fold
- PCA fit on training data only inside the loop for M4
- Expanding window from 2014-01 to 2024-12 (132 test months)

In [4]:
# Restrict to OOS window 2014-01 onward; the full panel (including pre-2014) is used as training data.
# PanelSplit with n_splits=132 puts the last 132 unique periods as test folds.
OOS_START = pd.Timestamp("2014-01-01")
N_OOS_MONTHS = panel_enc[panel_enc["date"] >= OOS_START]["date"].nunique()
print(f"OOS months: {N_OOS_MONTHS}  ({OOS_START.strftime('%Y-%m')} → {panel_enc.date.max().strftime('%Y-%m')})")

splitter = PanelSplit(periods=panel_enc["date"], n_splits=N_OOS_MONTHS, test_size=1)

def make_lgbm():
    return lgb.LGBMClassifier(
        class_weight="balanced",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbose=-1,
    )

MODELS = {
    "M0": FEATURES_M0,
    "M2": FEATURES_M2,
    "M3": FEATURES_M3,
    "M4": FEATURES_M4_BASE,   # PCA column appended dynamically below
}

prediction_rows = []

for fold_id, (train_idx, test_idx) in enumerate(splitter.split(), start=1):
    train_df = panel_enc.iloc[train_idx].copy()
    test_df  = panel_enc.iloc[test_idx].copy()
    test_month = test_df["date"].iloc[0]

    if fold_id % 12 == 1:
        print(f"  fold {fold_id:3d}/{N_OOS_MONTHS}  test={test_month.strftime('%Y-%m')}  "
              f"train_rows={len(train_df)}")

    for model_name, base_feats in MODELS.items():
        feats = list(base_feats)   # copy so we can append PCA col safely

        X_train = train_df[feats].copy()
        X_test  = test_df[feats].copy()

        # ── PCA for M4 (fit on training data only) ────────────────────────────
        if model_name == "M4":
            pca = PCA(n_components=1, random_state=42)
            pca.fit(X_train[PCA_INPUT_COLS].fillna(0))
            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train["pca_gt_1"] = pca.transform(X_train[PCA_INPUT_COLS].fillna(0))[:, 0]
            X_test["pca_gt_1"]  = pca.transform(X_test[PCA_INPUT_COLS].fillna(0))[:, 0]
            feats = feats + ["pca_gt_1"]

        y_train = train_df[TARGET]

        clf = make_lgbm()
        clf.fit(X_train, y_train)

        proba = clf.predict_proba(X_test)[:, 1]

        for i, row in enumerate(test_df.itertuples(index=False)):
            prediction_rows.append({
                "date":        test_month,
                "state_code":  row.state_code,
                "model":       model_name,
                "y_true":      int(getattr(row, TARGET)),
                "y_pred_proba": float(proba[i]),
            })

print(f"\nTotal prediction rows: {len(prediction_rows)}")

OOS months: 132  (2014-01 → 2024-12)
  fold   1/132  test=2014-01  train_rows=1683
  fold  13/132  test=2015-01  train_rows=2295
  fold  25/132  test=2016-01  train_rows=2907
  fold  37/132  test=2017-01  train_rows=3519
  fold  49/132  test=2018-01  train_rows=4131
  fold  61/132  test=2019-01  train_rows=4743
  fold  73/132  test=2020-01  train_rows=5355
  fold  85/132  test=2021-01  train_rows=5967
  fold  97/132  test=2022-01  train_rows=6579
  fold 109/132  test=2023-01  train_rows=7191
  fold 121/132  test=2024-01  train_rows=7803

Total prediction rows: 26928


In [5]:
# Save predictions
preds_df = pd.DataFrame(prediction_rows)
preds_df.to_parquet(OUT / "predictions.parquet", index=False)
print(f"Saved predictions.parquet  shape={preds_df.shape}")
print(preds_df.head())

Saved predictions.parquet  shape=(26928, 5)
        date state_code model  y_true  y_pred_proba
0 2014-01-01         AK    M0       0      0.000022
1 2014-01-01         AL    M0       0      0.000022
2 2014-01-01         AR    M0       0      0.000022
3 2014-01-01         AZ    M0       0      0.000022
4 2014-01-01         CA    M0       0      0.000022


In [6]:
# ── Pooled out-of-sample metrics ─────────────────────────────────────────────
print("=" * 55)
print(f"{'Model':<6}  {'AUC-ROC':>9}  {'AUC-PR':>9}  {'Shocks':>6}  {'N':>6}")
print("-" * 55)

for model_name in ["M0", "M2", "M3", "M4"]:
    sub = preds_df[preds_df["model"] == model_name]
    auc_roc = roc_auc_score(sub["y_true"], sub["y_pred_proba"])
    auc_pr  = average_precision_score(sub["y_true"], sub["y_pred_proba"])
    n_shocks = int(sub["y_true"].sum())
    print(f"{model_name:<6}  {auc_roc:9.4f}  {auc_pr:9.4f}  {n_shocks:6d}  {len(sub):6d}")

base_rate = preds_df[preds_df["model"] == "M0"]["y_true"].mean()
print("-" * 55)
print(f"No-skill AUC-PR baseline: {base_rate:.4f}  (empirical base rate)")

Model     AUC-ROC     AUC-PR  Shocks       N
-------------------------------------------------------
M0         0.6022     0.0198      97    6732
M2         0.8659     0.5189      97    6732
M3         0.8697     0.5161      97    6732
M4         0.9481     0.6536      97    6732
-------------------------------------------------------
No-skill AUC-PR baseline: 0.0144  (empirical base rate)
